In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import validation_curve
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
import joblib
import os
nltk.download('stopwords', quiet = True)
from nltk.corpus import stopwords
random_state = 42

In [ ]:
df = pd.read_csv('DUAT_dataset.csv')

In [ ]:
text_col = "texto"
stop_pt = stopwords.words('portuguese')

def scale_dataset(
    dataframe,
    scaler = None,
    types_col = "types_proporcao",
    num_palavras = "num_palavras",
    oversample = False
):
    numeric = dataframe.drop(columns = [text_col, types_col, num_palavras])

    feature_cols = numeric.columns[:-1]
    target_col = numeric.columns[-1]

    X = numeric[feature_cols].values
    y = numeric[target_col].values

    if scaler is None:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
    else:
        X_scaled = scaler.transform(X)

    X_scaled_df = pd.DataFrame(
    X_scaled, columns = feature_cols, index = dataframe.index)

    return X_scaled_df, y, scaler

In [ ]:
def embedding(
    df, idx_treino = None, coluna_texto = text_col,
    max_features = 300, min_df = 5, stop_words = stop_pt, tfidf = None
):

    if tfidf is None:
        tfidf = TfidfVectorizer(
        max_features = max_features, min_df = min_df, stop_words = stop_words)
        texto_fit = df.loc[idx_treino, coluna_texto] if idx_treino is not None else df[coluna_texto]
        tfidf.fit(texto_fit)

    X_tfidf = tfidf.transform(df[coluna_texto])
    tfidf_cols = [f"tfidf_{t}" for t in tfidf.get_feature_names_out()]
    df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_cols, index=df.index)

    return df_tfidf, tfidf

In [ ]:
train, valid, test = np.split(
df.sample(
frac = 1, random_state = random_state), [int(0.6*len(df)), int(0.8 * len(df))])

X_train_num, y_train, scaler = scale_dataset(train, scaler = None)
X_train_emb, tfidf = embedding(train, idx_treino = train.index)

X_train_num = X_train_num.reset_index(drop = True)
X_train_emb = X_train_emb.reset_index(drop = True)
X_train = pd.concat([X_train_num, X_train_emb], axis = 1)

X_valid_num, y_valid, _ = scale_dataset(valid, scaler = scaler)
X_valid_emb, _ = embedding(valid, tfidf=tfidf)

X_valid_num = X_valid_num.reset_index(drop = True)
X_valid_emb = X_valid_emb.reset_index(drop = True)
X_valid = pd.concat([X_valid_num, X_valid_emb], axis = 1)

X_test_num, y_test, _ = scale_dataset(test, scaler = scaler)
X_test_emb, _ = embedding(test, tfidf=tfidf)

X_test_num = X_test_num.reset_index(drop = True)
X_test_emb = X_test_emb.reset_index(drop = True)
X_test = pd.concat([X_test_num, X_test_emb], axis = 1)

/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
svm_model = SVC()
param_grid = {'C': [80], 'kernel': ['poly'], 'degree': [3], 'coef0': [1.0], 'gamma': ['auto']}
grid = GridSearchCV(
estimator = svm_model, param_grid = param_grid, cv = 5, scoring = 'accuracy')
svm_model = grid.fit(X_train, y_train)

print(grid.best_params_)

{'C': 80, 'coef0': 1.0, 'degree': 3, 'gamma': 'auto', 'kernel': 'poly'}


In [ ]:
os.makedirs("Pipelines", exist_ok=True)
joblib.dump(svm_model, "Pipelines/svm.joblib")

['Pipelines/svm.joblib']